In [1]:
# 安装TTA包
!pip install ttach

In [2]:
# 安装ResNeSt模型包
!pip install resnest --pre -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import ttach as tta
from resnest.torch import resnest50
from cutmix.cutmix import CutMix
from cutmix.utils import CutMixCrossEntropyLoss
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import KFold
from PIL import Image
import os
import matplotlib.pyplot as plt
import torchvision.models as models
from tqdm import tqdm

In [4]:
labels_dataframe = pd.read_csv('../leaves_data/train.csv')
leaves_labels = sorted(list(set(labels_dataframe['label'])))
n_classes = len(leaves_labels)
print(n_classes)
leaves_labels[:10]

176


['abies_concolor',
 'abies_nordmanniana',
 'acer_campestre',
 'acer_ginnala',
 'acer_griseum',
 'acer_negundo',
 'acer_palmatum',
 'acer_pensylvanicum',
 'acer_platanoides',
 'acer_pseudoplatanus']

In [5]:
# 把label转成对应的索引
class_to_num = dict(zip(leaves_labels, range(n_classes)))
class_to_num

{'abies_concolor': 0,
 'abies_nordmanniana': 1,
 'acer_campestre': 2,
 'acer_ginnala': 3,
 'acer_griseum': 4,
 'acer_negundo': 5,
 'acer_palmatum': 6,
 'acer_pensylvanicum': 7,
 'acer_platanoides': 8,
 'acer_pseudoplatanus': 9,
 'acer_rubrum': 10,
 'acer_saccharinum': 11,
 'acer_saccharum': 12,
 'aesculus_flava': 13,
 'aesculus_glabra': 14,
 'aesculus_hippocastamon': 15,
 'aesculus_pavi': 16,
 'ailanthus_altissima': 17,
 'albizia_julibrissin': 18,
 'amelanchier_arborea': 19,
 'amelanchier_canadensis': 20,
 'amelanchier_laevis': 21,
 'asimina_triloba': 22,
 'betula_alleghaniensis': 23,
 'betula_jacqemontii': 24,
 'betula_lenta': 25,
 'betula_nigra': 26,
 'betula_populifolia': 27,
 'broussonettia_papyrifera': 28,
 'carpinus_betulus': 29,
 'carpinus_caroliniana': 30,
 'carya_cordiformis': 31,
 'carya_glabra': 32,
 'carya_ovata': 33,
 'carya_tomentosa': 34,
 'castanea_dentata': 35,
 'catalpa_bignonioides': 36,
 'catalpa_speciosa': 37,
 'cedrus_atlantica': 38,
 'cedrus_deodara': 39,
 'cedru

In [6]:
# 索引对应标签，预测的时候使用
num_to_class = {v : k for k, v in class_to_num.items()}
num_to_class

{0: 'abies_concolor',
 1: 'abies_nordmanniana',
 2: 'acer_campestre',
 3: 'acer_ginnala',
 4: 'acer_griseum',
 5: 'acer_negundo',
 6: 'acer_palmatum',
 7: 'acer_pensylvanicum',
 8: 'acer_platanoides',
 9: 'acer_pseudoplatanus',
 10: 'acer_rubrum',
 11: 'acer_saccharinum',
 12: 'acer_saccharum',
 13: 'aesculus_flava',
 14: 'aesculus_glabra',
 15: 'aesculus_hippocastamon',
 16: 'aesculus_pavi',
 17: 'ailanthus_altissima',
 18: 'albizia_julibrissin',
 19: 'amelanchier_arborea',
 20: 'amelanchier_canadensis',
 21: 'amelanchier_laevis',
 22: 'asimina_triloba',
 23: 'betula_alleghaniensis',
 24: 'betula_jacqemontii',
 25: 'betula_lenta',
 26: 'betula_nigra',
 27: 'betula_populifolia',
 28: 'broussonettia_papyrifera',
 29: 'carpinus_betulus',
 30: 'carpinus_caroliniana',
 31: 'carya_cordiformis',
 32: 'carya_glabra',
 33: 'carya_ovata',
 34: 'carya_tomentosa',
 35: 'castanea_dentata',
 36: 'catalpa_bignonioides',
 37: 'catalpa_speciosa',
 38: 'cedrus_atlantica',
 39: 'cedrus_deodara',
 40: 'c

In [7]:
# 构建数据集
class LeavesData(Dataset):
    def __init__(self, csv_path, img_path, mode='train_valid', resize_height=224, resize_width=224, transform=None):
        """
        Args:
            csv_path (string): csv 文件路径
            img_path (string): 图像文件所在路径
            mode (string): 训练模式还是测试模式
            valid_ratio (float): 验证集比例
        """

        self.resize_height = resize_height
        self.resize_width = resize_width
        self.transform = transform
        self.img_path = img_path
        self.mode = mode

        # 利用pandas读取csv文件
        self.data_info = pd.read_csv(csv_path, header=None)  #header=None是将表头也作为可读数据
        # 计算 length
        self.data_len = len(self.data_info.index) - 1
        
        if mode == 'train_valid':
            # 第一列包含图像文件的名称
            self.train_valid_image = np.asarray(self.data_info.iloc[1:, 0])
            # 第二列是图像的label
            self.train_valid_label = np.asarray(self.data_info.iloc[1:, 1])
            self.image_arr = self.train_valid_image
            self.label_arr = self.train_valid_label
        elif mode == 'test':
            self.test_image = np.asarray(self.data_info.iloc[1:, 0])
            self.image_arr = self.test_image
            
        self.real_len = len(self.image_arr)

        print('Finished reading the {} set of Leaves Dataset ({} samples found)'.format(mode, self.real_len))

    def __getitem__(self, index):
        # 从image_arr中得到索引对应的文件名
        single_image_name = self.image_arr[index].split('/')[-1]

        # 读取图像文件
        img_as_img = Image.open(os.path.join(self.img_path, single_image_name))
        img_as_img = self.transform(img_as_img)
        
        if self.mode == 'test':
            return img_as_img
        else:
            label = self.label_arr[index]
            number_label = class_to_num[label]

            return img_as_img, number_label
        
    def __len__(self):
        return self.real_len

In [8]:
train_transform = transforms.Compose([
    # 随机裁剪图像，所得图像为原始面积的0.08到1之间，高宽比在3/4和4/3之间。
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0), ratio=(3.0 / 4.0, 4.0 / 3.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])])

val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])])

In [9]:
train_val_path = '../leaves_data/train.csv'
test_path = '../leaves_data/test.csv'
img_path = '../input/classify-leaves/raw_images'

train_valid_dataset = LeavesData(train_val_path, img_path,  mode='train_valid')
test_dataset = LeavesData(test_path, img_path, mode='test')
print(train_valid_dataset.real_len)
print(test_dataset.real_len)

Finished reading the train_valid set of Leaves Dataset (18353 samples found)
Finished reading the test set of Leaves Dataset (8800 samples found)
18353
8800


In [10]:
# 用于是否要冻住模型的特征提取层
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        model = model
        for param in model.parameters():
            param.requires_grad = False

# ResNeSt模型
def resnest_model(num_classes, feature_extract = False):
    model_ft = resnest50(pretrained=True)
    set_parameter_requires_grad(model_ft, feature_extract)
    num_ftrs = model_ft.fc.in_features
    model_ft.fc = nn.Sequential(nn.Linear(num_ftrs, num_classes))

    return model_ft

In [11]:
def get_device():
    return 'cuda' if torch.cuda.is_available() else 'cpu'

device = get_device()
print(device)

cuda


In [12]:
# 设置超参数
k_folds = 5
num_epochs = 30
learning_rate = 1e-4
weight_decay = 1e-3
# 适配CutMix，接收软加权标签，正确算loss。
train_loss_function = CutMixCrossEntropyLoss(True)
valid_loss_function = nn.CrossEntropyLoss()
# 接收5折交叉验证结果
results = {}
torch.manual_seed(42)

kfold = KFold(n_splits=k_folds, shuffle=True)

In [ ]:
# K-fold Cross Validation model evaluation
for fold, (train_ids,valid_ids) in enumerate(kfold.split(train_valid_dataset)):
    print(f'FOLD {fold + 1}')
    print('--------------------------------------')

    # 随机打乱指定索引的样本
    train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
    valid_subsampler = torch.utils.data.SubsetRandomSampler(valid_ids)
    # 做CutMix混合数据增强：训练时随机把两张图裁剪拼接，标签按面积比例变成软标签。
    trainloader = torch.utils.data.DataLoader(
                          CutMix(LeavesData(train_val_path, img_path,  mode='train_valid', transform = train_transform), 
                                 num_class=176, beta=1.0, prob=0.5, num_mix=2), 
                                 batch_size=128, sampler=train_subsampler, num_workers=0)
    validloader = torch.utils.data.DataLoader(
                            LeavesData(train_val_path, img_path, mode='train_valid', transform = val_test_transform),
                                       batch_size=128, sampler=valid_subsampler, num_workers=0)

    model = resnest_model(176)
    model = model.to(device)
    # AdamW把权重衰减和梯度更新拆开独立计算，避免正则化效果被自适应学习率干扰，泛化更好。
    optimizer = torch.optim.AdamW(model.parameters(),lr=learning_rate, weight_decay=weight_decay)
    # 训练学习率按余弦曲线平滑下降：前期 lr大，快速收敛；后期lr小，精细微调权重，容易找到最优最低点；
    # 到达周期末尾会重置lr重新下降，避免卡在局部最优。
    scheduler = CosineAnnealingLR(optimizer, T_max=10)
    best_val_acc = 0.0
    best_save_path = f'./resnest-model-fold-{fold}.pth'
    
    for epoch in range(0,num_epochs):
        model.train()
        print(f'Starting epoch {epoch+1}')
        train_losses = []
        train_accs = []

        for batch in tqdm(trainloader):
            imgs, labels = batch
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = train_loss_function(logits,labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            acc = (logits.argmax(dim=-1) == labels).float().mean()
            train_losses.append(loss.item())
            train_accs.append(acc)
        print("Learning rate of Epoch %d: %.3f" % (epoch+1,optimizer.param_groups[0]['lr']))
        scheduler.step()
        train_loss = sum(train_losses) / len(train_losses)
        train_acc = sum(train_accs) / len(train_accs)
        print(f"[ Train | {epoch + 1:03d}/{num_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")
    
        # Start Validation
        model.eval()
        valid_losses = []
        valid_accs = []
        with torch.no_grad():
            for batch in tqdm(validloader):
                imgs, labels = batch
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                loss = valid_loss_function(logits, labels)
                acc = (logits.argmax(dim=-1) == labels).float().mean()
                valid_losses.append(loss.item())        
                valid_accs.append(acc)
                
            valid_loss = sum(valid_losses)/len(valid_losses)
            valid_acc = sum(valid_accs)/len(valid_accs)
            print(f"[ Valid | {epoch + 1:03d}/{num_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")
            
            if valid_acc > best_val_acc:
                best_val_acc = valid_acc
                torch.save(model.state_dict(), best_save_path)
                print(f"New best model found for Fold {fold}, val_acc = {best_val_acc:.4f}; saved successfully")
                
    print(f'Training of Fold {fold} completed. Best validation accuracy of this fold: {best_val_acc:.4f}')
    results[fold] = best_val_acc
    print('--------------------------------------')

# Print fold results
print(f'K-FOLD CROSS VALIDATION RESULTS FOR {k_folds} FOLDS')
print('-----------------------------------')
total_summation = 0.0
for key, value in results.items():
    print(f'Fold {key}: {value} ')
    total_summation += value
print(f'Average accuracy: {total_summation/len(results.items())}')

FOLD 1
--------------------------------------
Finished reading the train_valid set of Leaves Dataset (18353 samples found)
Finished reading the train_valid set of Leaves Dataset (18353 samples found)


Downloading: "https://github.com/zhanghang1989/ResNeSt/releases/download/weights_step1/resnest50-528c19ca.pth" to C:\Users\OMEN/.cache\torch\hub\checkpoints\resnest50-528c19ca.pth


  0%|          | 0.00/105M [00:00<?, ?B/s]

In [ ]:
testloader = torch.utils.data.DataLoader(LeavesData(test_path, img_path,  mode='test', transform = val_test_transform),
                                         batch_size=32, num_workers=0)  # 不做打乱

In [ ]:
# predict
all_logits = None
for test_fold in range(k_folds):
    # 每折重新创建模型，释放上一折权重
    model = resnest_model(176).to(device)
    model_path = f'./resnest-model-fold-{test_fold}.pth'
    model.load_state_dict(torch.load(model_path))

    model.eval()
    # 解释tta：给分类模型套一层TTA包装器，tta.aliases.five_crop_transform(200,200)：使用「五裁剪」增强策略：
    # 对单张原图，分别裁剪：左上、右上、左下、右下、中心 5 个 200×200 区域，再加上原图翻转，一共生成多张变体图。
    # 推理时用五裁剪多图投票，最后平均所有预测概率作为最终输出，降低单张图裁剪位置带来的误差，提升测试集准确率。
    tta_model = tta.ClassificationTTAWrapper(model, tta.aliases.five_crop_transform(200,200)) # Test-Time Augmentation

    fold_logits = []
    for batch in tqdm(testloader):
        imgs = batch
        with torch.no_grad():
            logits = tta_model(imgs.to(device))
        fold_logits.append(logits.cpu())
        
    fold_logits = torch.cat(fold_logits, dim=0)
    # 累加
    if all_logits is None:
        all_logits = fold_logits
    else:
        all_logits += fold_logits

# 全部5折跑完，求平均
all_logits = all_logits / 5.0
final_pred_idx = all_logits.argmax(dim=-1).numpy().tolist()

# 数字索引转回类别字符串
preds = [num_to_class[i] for i in final_pred_idx]

test_data = pd.read_csv(test_path)
test_data['label'] = pd.Series(preds)
prediction = pd.concat([test_data['image'], test_data['label']], axis=1)
prediction.to_csv('prediction_ResNeSt.csv', index=False)
print("ResNeSt Model Results Done.")

In [ ]:
# ResNeXt模型
# resnext50_32x4d模型
def resnext_model(num_classes, feature_extract = False, use_pretrained=True):
    model_ft = models.resnext50_32x4d(pretrained=use_pretrained)
    set_parameter_requires_grad(model_ft, feature_extract)
    num_ftrs = model_ft.fc.in_features
    model_ft.fc = nn.Sequential(nn.Linear(num_ftrs, num_classes))

    return model_ft

In [ ]:
# 设置超参数
k_folds = 5
num_epochs = 30
learning_rate = 1e-3
weight_decay = 1e-3
# 适配CutMix，接收软加权标签，正确算loss。
train_loss_function = CutMixCrossEntropyLoss(True)
valid_loss_function = nn.CrossEntropyLoss()
# 接收5折交叉验证结果
results = {}
torch.manual_seed(42)

kfold = KFold(n_splits=k_folds, shuffle=True)

In [ ]:
# K-fold Cross Validation model evaluation
for fold, (train_ids,valid_ids) in enumerate(kfold.split(train_valid_dataset)):
    print(f'FOLD {fold + 1}')
    print('--------------------------------------')

    # 随机打乱指定索引的样本
    train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
    valid_subsampler = torch.utils.data.SubsetRandomSampler(valid_ids)
    # 做CutMix混合数据增强：训练时随机把两张图裁剪拼接，标签按面积比例变成软标签。
    trainloader = torch.utils.data.DataLoader(
                          CutMix(LeavesData(train_val_path, img_path,  mode='train_valid', transform = train_transform), 
                                 num_class=176, beta=1.0, prob=0.5, num_mix=2), 
                                 batch_size=128, sampler=train_subsampler, num_workers=0)
    validloader = torch.utils.data.DataLoader(
                            LeavesData(train_val_path, img_path, mode='train_valid', transform = val_test_transform),
                                       batch_size=128, sampler=valid_subsampler, num_workers=0)

    model = resnext_model(176)
    model = model.to(device)
    # AdamW把权重衰减和梯度更新拆开独立计算，避免正则化效果被自适应学习率干扰，泛化更好。
    optimizer = torch.optim.AdamW(model.parameters(),lr=learning_rate, weight_decay=weight_decay)
    # 训练学习率按余弦曲线平滑下降：前期 lr大，快速收敛；后期lr小，精细微调权重，容易找到最优最低点；
    # 到达周期末尾会重置lr重新下降，避免卡在局部最优。
    scheduler = CosineAnnealingLR(optimizer, T_max=10)
    best_val_acc = 0.0
    best_save_path = f'./resnext-model-fold-{fold}.pth'
    
    for epoch in range(0,num_epochs):
        model.train()
        print(f'Starting epoch {epoch+1}')
        train_losses = []
        train_accs = []

        for batch in tqdm(trainloader):
            imgs, labels = batch
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = train_loss_function(logits,labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            acc = (logits.argmax(dim=-1) == labels).float().mean()
            train_losses.append(loss.item())
            train_accs.append(acc)
        print("Learning rate of Epoch %d: %.3f" % (epoch+1,optimizer.param_groups[0]['lr']))
        scheduler.step()
        train_loss = sum(train_losses) / len(train_losses)
        train_acc = sum(train_accs) / len(train_accs)
        print(f"[ Train | {epoch + 1:03d}/{num_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")
    
        # Start Validation
        model.eval()
        valid_losses = []
        valid_accs = []
        with torch.no_grad():
            for batch in tqdm(validloader):
                imgs, labels = batch
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                loss = valid_loss_function(logits, labels)
                acc = (logits.argmax(dim=-1) == labels).float().mean()
                valid_losses.append(loss.item())        
                valid_accs.append(acc)
                
            valid_loss = sum(valid_losses)/len(valid_losses)
            valid_acc = sum(valid_accs)/len(valid_accs)
            print(f"[ Valid | {epoch + 1:03d}/{num_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")
            
            if valid_acc > best_val_acc:
                best_val_acc = valid_acc
                torch.save(model.state_dict(), best_save_path)
                print(f"New best model found for Fold {fold}, val_acc = {best_val_acc:.4f}; saved successfully")
                
    print(f'Training of Fold {fold} completed. Best validation accuracy of this fold: {best_val_acc:.4f}')
    results[fold] = best_val_acc
    print('--------------------------------------')

# Print fold results
print(f'K-FOLD CROSS VALIDATION RESULTS FOR {k_folds} FOLDS')
print('-----------------------------------')
total_summation = 0.0
for key, value in results.items():
    print(f'Fold {key}: {value} ')
    total_summation += value
print(f'Average accuracy: {total_summation/len(results.items())}')

In [ ]:
testloader = torch.utils.data.DataLoader(LeavesData(test_path, img_path,  mode='test', transform = val_test_transform),
                                         batch_size=128, num_workers=0)

In [ ]:
# predict
all_logits = None
for test_fold in range(k_folds):
    # 每折重新创建模型，释放上一折权重
    model = resnext_model(176).to(device)
    model_path = f'./resnext-model-fold-{test_fold}.pth'
    model.load_state_dict(torch.load(model_path))

    model.eval()
    # 解释tta：给分类模型套一层TTA包装器，tta.aliases.five_crop_transform(200,200)：使用「五裁剪」增强策略：
    # 对单张原图，分别裁剪：左上、右上、左下、右下、中心 5 个 200×200 区域，再加上原图翻转，一共生成多张变体图。
    # 推理时用五裁剪多图投票，最后平均所有预测概率作为最终输出，降低单张图裁剪位置带来的误差，提升测试集准确率。
    tta_model = tta.ClassificationTTAWrapper(model, tta.aliases.five_crop_transform(200,200)) # Test-Time Augmentation

    fold_logits = []
    for batch in tqdm(testloader):
        imgs = batch
        with torch.no_grad():
            logits = tta_model(imgs.to(device))
        fold_logits.append(logits.cpu())
        
    fold_logits = torch.cat(fold_logits, dim=0)
    # 累加
    if all_logits is None:
        all_logits = fold_logits
    else:
        all_logits += fold_logits

# 全部5折跑完，求平均
all_logits = all_logits / 5.0
final_pred_idx = all_logits.argmax(dim=-1).numpy().tolist()

# 数字索引转回类别字符串
preds = [num_to_class[i] for i in final_pred_idx]

test_data = pd.read_csv(test_path)
test_data['label'] = pd.Series(preds)
prediction = pd.concat([test_data['image'], test_data['label']], axis=1)
prediction.to_csv('prediction_ResNeXt.csv', index=False)
print("ResNeXt Model Results Done.")

In [ ]:
# densenet161模型
def dense_model(num_classes, feature_extract = False, use_pretrained=True):

    model_ft = models.densenet161(pretrained=use_pretrained)
    set_parameter_requires_grad(model_ft, feature_extract)
    num_ftrs = model_ft.classifier.in_features
    model_ft.classifier = nn.Sequential(nn.Linear(num_ftrs, num_classes))

    return model_ft

In [ ]:
# 设置超参数
k_folds = 5
num_epochs = 30
learning_rate = 1e-4
weight_decay = 1e-3
# 适配CutMix，接收软加权标签，正确算loss。
train_loss_function = CutMixCrossEntropyLoss(True)
valid_loss_function = nn.CrossEntropyLoss()
# 接收5折交叉验证结果
results = {}
torch.manual_seed(42)

kfold = KFold(n_splits=k_folds, shuffle=True)

In [ ]:
# K-fold Cross Validation model evaluation
for fold, (train_ids,valid_ids) in enumerate(kfold.split(train_valid_dataset)):
    print(f'FOLD {fold + 1}')
    print('--------------------------------------')

    # 随机打乱指定索引的样本
    train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
    valid_subsampler = torch.utils.data.SubsetRandomSampler(valid_ids)
    # 做CutMix混合数据增强：训练时随机把两张图裁剪拼接，标签按面积比例变成软标签。
    trainloader = torch.utils.data.DataLoader(
                          CutMix(LeavesData(train_val_path, img_path,  mode='train_valid', transform = train_transform), 
                                 num_class=176, beta=1.0, prob=0.5, num_mix=2), 
                                 batch_size=32, sampler=train_subsampler, num_workers=0)
    validloader = torch.utils.data.DataLoader(
                            LeavesData(train_val_path, img_path, mode='train_valid', transform = val_test_transform),
                                       batch_size=32, sampler=valid_subsampler, num_workers=0)

    model = dense_model(176)
    model = model.to(device)
    # AdamW把权重衰减和梯度更新拆开独立计算，避免正则化效果被自适应学习率干扰，泛化更好。
    optimizer = torch.optim.AdamW(model.parameters(),lr=learning_rate, weight_decay=weight_decay)
    # 训练学习率按余弦曲线平滑下降：前期 lr大，快速收敛；后期lr小，精细微调权重，容易找到最优最低点；
    # 到达周期末尾会重置lr重新下降，避免卡在局部最优。
    scheduler = CosineAnnealingLR(optimizer, T_max=10)
    best_val_acc = 0.0
    best_save_path = f'./dense-model-fold-{fold}.pth'
    
    for epoch in range(0,num_epochs):
        model.train()
        print(f'Starting epoch {epoch+1}')
        train_losses = []
        train_accs = []

        for batch in tqdm(trainloader):
            imgs, labels = batch
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = train_loss_function(logits,labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            acc = (logits.argmax(dim=-1) == labels).float().mean()
            train_losses.append(loss.item())
            train_accs.append(acc)
        print("Learning rate of Epoch %d: %.3f" % (epoch+1,optimizer.param_groups[0]['lr']))
        scheduler.step()
        train_loss = sum(train_losses) / len(train_losses)
        train_acc = sum(train_accs) / len(train_accs)
        print(f"[ Train | {epoch + 1:03d}/{num_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")
    
        # Start Validation
        model.eval()
        valid_losses = []
        valid_accs = []
        with torch.no_grad():
            for batch in tqdm(validloader):
                imgs, labels = batch
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                loss = valid_loss_function(logits, labels)
                acc = (logits.argmax(dim=-1) == labels).float().mean()
                valid_losses.append(loss.item())        
                valid_accs.append(acc)
                
            valid_loss = sum(valid_losses)/len(valid_losses)
            valid_acc = sum(valid_accs)/len(valid_accs)
            print(f"[ Valid | {epoch + 1:03d}/{num_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")
            
            if valid_acc > best_val_acc:
                best_val_acc = valid_acc
                torch.save(model.state_dict(), best_save_path)
                print(f"New best model found for Fold {fold}, val_acc = {best_val_acc:.4f}; saved successfully")
                
    print(f'Training of Fold {fold} completed. Best validation accuracy of this fold: {best_val_acc:.4f}')
    results[fold] = best_val_acc
    print('--------------------------------------')

# Print fold results
print(f'K-FOLD CROSS VALIDATION RESULTS FOR {k_folds} FOLDS')
print('-----------------------------------')
total_summation = 0.0
for key, value in results.items():
    print(f'Fold {key}: {value} ')
    total_summation += value
print(f'Average accuracy: {total_summation/len(results.items())}')

In [ ]:
testloader = torch.utils.data.DataLoader(LeavesData(test_path, img_path,  mode='test', transform = val_test_transform),
                                         batch_size=32, num_workers=0)

In [ ]:
# predict
all_logits = None
for test_fold in range(k_folds):
    # 每折重新创建模型，释放上一折权重
    model = dense_model(176).to(device)
    model_path = f'./dense-model-fold-{test_fold}.pth'
    model.load_state_dict(torch.load(model_path))

    model.eval()
    # 解释tta：给分类模型套一层TTA包装器，tta.aliases.five_crop_transform(200,200)：使用「五裁剪」增强策略：
    # 对单张原图，分别裁剪：左上、右上、左下、右下、中心 5 个 200×200 区域，再加上原图翻转，一共生成多张变体图。
    # 推理时用五裁剪多图投票，最后平均所有预测概率作为最终输出，降低单张图裁剪位置带来的误差，提升测试集准确率。
    tta_model = tta.ClassificationTTAWrapper(model, tta.aliases.five_crop_transform(200,200)) # Test-Time Augmentation

    fold_logits = []
    for batch in tqdm(testloader):
        imgs = batch
        with torch.no_grad():
            logits = tta_model(imgs.to(device))
        fold_logits.append(logits.cpu())
        
    fold_logits = torch.cat(fold_logits, dim=0)
    # 累加
    if all_logits is None:
        all_logits = fold_logits
    else:
        all_logits += fold_logits

# 全部5折跑完，求平均
all_logits = all_logits / 5.0
final_pred_idx = all_logits.argmax(dim=-1).numpy().tolist()

# 数字索引转回类别字符串
preds = [num_to_class[i] for i in final_pred_idx]

test_data = pd.read_csv(test_path)
test_data['label'] = pd.Series(preds)
prediction = pd.concat([test_data['image'], test_data['label']], axis=1)
prediction.to_csv('prediction_Dense.csv', index=False)
print("Dense Model Results Done.")

In [ ]:
df_resnest = pd.read_csv('./prediction_ResNeSt.csv')
df_resnext = pd.read_csv('../prediction_ResNeXt.csv')
df_densenet = pd.read_csv('../prediction_Dense.csv')

In [ ]:
df_all = df_resnest.copy()
df_all.rename(columns = {'label':'label_resnest'},inplace=True)
df_all['label_resnext'] = df_resnext.copy()['label']
df_all['label_densenet'] = df_densenet.copy()['label']
df_all.head()

In [ ]:
df_all['label']=0
for rows in range(len(df_all)):
    if (df_all['label_resnest'].iloc[rows]==df_all['label_resnext'].iloc[rows]) or (df_all['label_resnest'].iloc[rows]==df_all['label_densenet'].iloc[rows]):
        df_all['label'].iloc[rows] = df_all.copy()['label_resnest'].iloc[rows]
    elif df_all['label_resnext'].iloc[rows]==df_all['label_densenet'].iloc[rows]:
        df_all['label'].iloc[rows] = df_all.copy()['label_resnext'].iloc[rows]
    else:
        df_all['label'].iloc[rows] = df_all.copy()['label_resnest'].iloc[rows]
df_all.head()

In [ ]:
df_final = df_all.copy()[['image','label']]
df_final.head()

In [ ]:
df_final.to_csv('./ensemble-prediction.csv', index=False)
print('Final results successfully saved!')